# Prep one RFROMV2.3 netCDF for NODD and upload it

This notebook prepares **one** output netCDF for the NOAA Open Data Dissemination (NODD)
GCP bucket and uploads it, so the workflow can be tested before scaling to all files
(GitHub issue #1).

**What it does**

1. Build the list of output files ("blocks" of 100 time steps) and cross-walk each block
   to the monthly source files on ERDDAP.
2. Download the monthly source netCDFs for one block from ERDDAP to a scratch directory.
3. Open + combine them and select exactly the block's 100 time steps.
4. Fix metadata for CF-compliance.
5. Rechunk to `(time=100, mean_pressure=1, latitude=180, longitude=180)` (~13 MB per chunk).
6. Write the output netCDF locally with lossless compression (`zlib` level 4 + shuffle).
7. Upload that single file to the NODD bucket.

The output files are also designed to **virtualize cleanly into an Icechunk store later**
(uniform chunk grid across files, codec-representable compression, consistent dtype/fill) —
see the notes in Step 5.

**Source dataset** (ERDDAP griddap id `argo_rfromv23_temp`): `ocean_temperature(time, mean_pressure, latitude, longitude)`,
float32, dims `time=1670, mean_pressure=58, latitude=720, longitude=1440`. Monthly source files hold
4–5 seven-day time steps and are unchunked.

> **Disk needed:** one block downloads ~23–25 monthly files (~1.2 GB each ≈ 30 GB). The
> output is ~24 GB uncompressed, but with compression on it should land around ~6–10 GB.
> Allow ~40 GB free on the scratch directory. A local disk (or S3 scratch) is fine; the
> monthly downloads can be deleted afterward (last cell).

## Setup

In [1]:
import os
import shutil

import numpy as np
import pandas as pd
import requests
import xarray as xr
import gcsfs

## Configuration

Everything that changes between variables (temp vs. sal) or between test/production lives here.

In [2]:
# --- Source (ERDDAP) -------------------------------------------------------
DATASET_ID = "argo_rfromv23_temp"          # ERDDAP griddap dataset id
DATA_VAR   = "ocean_temperature"            # main data variable in the files
MONTHLY_PREFIX = "RFROMV23_TEMP_STABLE"     # monthly source file prefix on ERDDAP

FILES_URL = f"https://data.pmel.noaa.gov/pmel/erddap/files/{DATASET_ID}"
TIME_CSV  = f"https://data.pmel.noaa.gov/pmel/erddap/griddap/{DATASET_ID}.csv?time"

# --- Output layout ---------------------------------------------------------
OUT_PREFIX = "RFROMV23_TEMP_STABLE"         # output file prefix (NODD naming)
BLOCK_SIZE = 100                            # time steps per output file
BLOCK_INDEX = 0                             # <-- which block to process/upload in this test

# Physical chunk sizes, in variable dim order (time, mean_pressure, latitude, longitude).
# 100 * 1 * 180 * 180 * 4 bytes = ~12.96 MB per chunk.
CHUNKS = {"time": 100, "mean_pressure": 1, "latitude": 180, "longitude": 180}

# --- Scratch / local paths -------------------------------------------------
SCRATCH_DIR = "/home/jovyan/shared-public/rfromv-scratch"   # download + output dir
DOWNLOAD_DIR = os.path.join(SCRATCH_DIR, "erddap")
OUTPUT_DIR   = os.path.join(SCRATCH_DIR, "nodd")
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- NODD (GCP) destination ------------------------------------------------
GCS_TOKEN = "/home/jovyan/.config/gcloud/application_default_credentials.json"
NODD_BUCKET = "noaa-oar-rfrom"
NODD_NETCDF_DIR = "netcdfs"                 # <-- confirm the exact dir name in the bucket
NODD_DEST = f"gs://{NODD_BUCKET}/{NODD_NETCDF_DIR}"

## Step 1 — Build the file blocks

Read the full time axis from ERDDAP, split it into blocks of `BLOCK_SIZE` time steps, and
cross-walk each block to the monthly source files that cover it. The output filename encodes
the first and last date in the block.

In [ ]:
def make_file_blocks(times, block_size=BLOCK_SIZE):
    """Split the time index into blocks and map each to its monthly source files."""
    times = pd.DatetimeIndex(times)
    blocks = []
    for i in range(0, len(times), block_size):
        block_times = times[i:i + block_size]
        months = block_times.to_period("M").unique()
        urls = [
            f"{FILES_URL}/{MONTHLY_PREFIX}_{m.year}_{m.month:02d}.nc"
            for m in months
        ]
        start, end = block_times[0], block_times[-1]
        filename = f"{OUT_PREFIX}_{start:%Y-%m-%d}_{end:%Y-%m-%d}.nc"
        blocks.append({
            "block": i // block_size,
            "start": start,
            "end": end,
            "n_times": len(block_times),
            "filename": filename,
            "urls": urls,
        })
    return blocks


# Read the time axis (row 0 is the header, row 1 is the units row -> skip it).
# ERDDAP returns tz-aware UTC times (trailing "Z"), but the netCDF files decode to
# tz-naive datetimes; strip the tz so the two are comparable when we slice in Step 3.
times = pd.read_csv(TIME_CSV, skiprows=[1])
times["time"] = pd.to_datetime(times["time"], utc=True).dt.tz_localize(None)
times = pd.DatetimeIndex(times["time"])

blocks = make_file_blocks(times)
print(f"{len(times)} time steps -> {len(blocks)} output files\n")
for b in blocks:
    print(f"[{b['block']:2d}] {b['filename']}  ({b['n_times']} steps, {len(b['urls'])} monthly files)")

In [4]:
# The single block we will process and upload in this test
block = blocks[BLOCK_INDEX]
print("Processing block", block["block"])
print("Output file:", block["filename"])
print("Time range :", block["start"].date(), "->", block["end"].date(), f"({block['n_times']} steps)")
print("Source files:")
for u in block["urls"]:
    print("  ", u.split("/")[-1])

Processing block 0
Output file: RFROMV23_TEMP_STABLE_1993-01-01_1994-11-25.nc
Time range : 1993-01-01 -> 1994-11-25 (100 steps)
Source files:
   RFROMV23_TEMP_STABLE_1993_01.nc
   RFROMV23_TEMP_STABLE_1993_02.nc
   RFROMV23_TEMP_STABLE_1993_03.nc
   RFROMV23_TEMP_STABLE_1993_04.nc
   RFROMV23_TEMP_STABLE_1993_05.nc
   RFROMV23_TEMP_STABLE_1993_06.nc
   RFROMV23_TEMP_STABLE_1993_07.nc
   RFROMV23_TEMP_STABLE_1993_08.nc
   RFROMV23_TEMP_STABLE_1993_09.nc
   RFROMV23_TEMP_STABLE_1993_10.nc
   RFROMV23_TEMP_STABLE_1993_11.nc
   RFROMV23_TEMP_STABLE_1993_12.nc
   RFROMV23_TEMP_STABLE_1994_01.nc
   RFROMV23_TEMP_STABLE_1994_02.nc
   RFROMV23_TEMP_STABLE_1994_03.nc
   RFROMV23_TEMP_STABLE_1994_04.nc
   RFROMV23_TEMP_STABLE_1994_05.nc
   RFROMV23_TEMP_STABLE_1994_06.nc
   RFROMV23_TEMP_STABLE_1994_07.nc
   RFROMV23_TEMP_STABLE_1994_08.nc
   RFROMV23_TEMP_STABLE_1994_09.nc
   RFROMV23_TEMP_STABLE_1994_10.nc
   RFROMV23_TEMP_STABLE_1994_11.nc


## Step 2 — Download the monthly source files for this block

Stream each monthly file to `DOWNLOAD_DIR`. Files already present with the right size are
skipped, so the cell is safe to re-run.

In [5]:
%%time
def download(url, dest, chunk=16 * 1024 * 1024):
    """Download url -> dest, skipping if a complete copy already exists."""
    head = requests.head(url, timeout=60)
    head.raise_for_status()
    remote_size = int(head.headers.get("Content-Length", 0))
    if os.path.exists(dest) and remote_size and os.path.getsize(dest) == remote_size:
        print(f"  skip (have) {os.path.basename(dest)}")
        return dest
    tmp = dest + ".part"
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for block_bytes in r.iter_content(chunk_size=chunk):
                f.write(block_bytes)
    os.replace(tmp, dest)
    print(f"  got  {os.path.basename(dest)}  ({os.path.getsize(dest) / 1e9:.2f} GB)")
    return dest


local_files = []
for url in block["urls"]:
    dest = os.path.join(DOWNLOAD_DIR, url.split("/")[-1])
    local_files.append(download(url, dest))

total_gb = sum(os.path.getsize(f) for f in local_files) / 1e9
print(f"\nDownloaded {len(local_files)} files, {total_gb:.1f} GB total")

  got  RFROMV23_TEMP_STABLE_1993_01.nc  (1.20 GB)
  got  RFROMV23_TEMP_STABLE_1993_02.nc  (0.96 GB)
  got  RFROMV23_TEMP_STABLE_1993_03.nc  (0.96 GB)
  got  RFROMV23_TEMP_STABLE_1993_04.nc  (1.20 GB)
  got  RFROMV23_TEMP_STABLE_1993_05.nc  (0.96 GB)
  got  RFROMV23_TEMP_STABLE_1993_06.nc  (0.96 GB)
  got  RFROMV23_TEMP_STABLE_1993_07.nc  (1.20 GB)
  got  RFROMV23_TEMP_STABLE_1993_08.nc  (0.96 GB)
  got  RFROMV23_TEMP_STABLE_1993_09.nc  (0.96 GB)
  got  RFROMV23_TEMP_STABLE_1993_10.nc  (1.20 GB)
  got  RFROMV23_TEMP_STABLE_1993_11.nc  (0.96 GB)
  got  RFROMV23_TEMP_STABLE_1993_12.nc  (1.20 GB)
  got  RFROMV23_TEMP_STABLE_1994_01.nc  (0.96 GB)
  got  RFROMV23_TEMP_STABLE_1994_02.nc  (0.96 GB)
  got  RFROMV23_TEMP_STABLE_1994_03.nc  (0.96 GB)
  got  RFROMV23_TEMP_STABLE_1994_04.nc  (1.20 GB)
  got  RFROMV23_TEMP_STABLE_1994_05.nc  (0.96 GB)
  got  RFROMV23_TEMP_STABLE_1994_06.nc  (0.96 GB)
  got  RFROMV23_TEMP_STABLE_1994_07.nc  (1.20 GB)
  got  RFROMV23_TEMP_STABLE_1994_08.nc  (0.96 GB)


## Step 3 — Open, combine, and select the block's time steps

The monthly files at the block boundaries can contain a few time steps outside the block, so
we slice to exactly `[start, end]` — this returns exactly the block's time steps.

In [ ]:
# Open with `data_vars="minimal"` so only variables that already have a `time` dim are
# concatenated: this keeps `mean_pressure_bnds` as (mean_pressure, vertices) instead of
# broadcasting it along time (i.e. it does NOT alter the original data). `chunks={"mean_pressure": 1}`
# makes each dask read a whole contiguous lat/lon plane at one pressure level, which is the
# file's natural layout — this is what keeps the Step 6 write from thrashing on strided I/O.
ds = xr.open_mfdataset(
    local_files,
    engine="h5netcdf",
    combine="by_coords",     # order by the time coordinate in each file
    data_vars="minimal",     # don't add a spurious time dim to mean_pressure_bnds
    coords="minimal",
    compat="override",       # coords/bounds are identical across files (verified); take the first
    chunks={"mean_pressure": 1},   # read contiguous spatial planes; rechunk in Step 5
)

# Trim to exactly this block's time steps
ds = ds.sel(time=slice(block["start"], block["end"]))

assert ds.sizes["time"] == block["n_times"], (
    f"expected {block['n_times']} times, got {ds.sizes['time']}"
)
print("Selected dims:", dict(ds.sizes))
ds

## Step 4 — Fix metadata (CF-compliance)

Preserve the source global attributes and close the gaps needed for CF-compliance:

- add `standard_name` to the data variable and to `mean_pressure`,
- mark the vertical coordinate with `positive="down"` and `axis="Z"` (and `axis` on lat/lon/time),
- set `Conventions = "CF-1.10, ACDD-1.3"` and append a processing note to `history`,
- drop the stale, self-contradictory `Description` on `time`.

> The `standard_name` values are **inferred** from the source descriptions (conservative
> temperature → `sea_water_conservative_temperature`, pressure → `sea_water_pressure`).
> Confirm them against the CF standard-name table. After writing, you can validate with the
> IOOS `compliance-checker` (`compliance-checker --test=cf <file>`) if it's installed.

In [ ]:
stamp = pd.Timestamp.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")


def clean_utf8(v):
    """Repair source string attributes that were read as lone surrogates.

    Some source attrs (e.g. `references`, which contains an en-dash) hold valid
    UTF-8 *bytes* that xarray decoded with surrogateescape, producing lone
    surrogates that h5netcdf then refuses to write. Re-encode to the original
    bytes and decode as proper UTF-8.
    """
    if isinstance(v, str):
        return v.encode("utf-8", "surrogateescape").decode("utf-8", "replace")
    return v


# --- CF-compliant variable attributes --------------------------------------
# standard_name assignments below are inferred from the source descriptions;
# please confirm them against the CF standard-name table:
#   ocean_temperature -> TEOS-10 conservative temperature
#   mean_pressure     -> sea-water pressure (vertical axis, positive down)
ds[DATA_VAR].attrs.update({
    "standard_name": "sea_water_conservative_temperature",
    "units": "degree_Celsius",
})
ds["mean_pressure"].attrs.update({
    "standard_name": "sea_water_pressure",
    "units": "decibar",
    "positive": "down",
    "axis": "Z",
})
ds["latitude"].attrs.update({"standard_name": "latitude", "units": "degrees_north", "axis": "Y"})
ds["longitude"].attrs.update({"standard_name": "longitude", "units": "degrees_east", "axis": "X"})
ds["time"].attrs.update({"standard_name": "time", "axis": "T", "long_name": "Time"})
# The source carries a stale, contradictory time "Description" ("days since 1950");
# the real encoding is seconds since 1970 (set in Step 5), so drop it to avoid confusion.
ds["time"].attrs.pop("Description", None)

# --- Global attributes ------------------------------------------------------
ds.attrs["Conventions"] = "CF-1.10, ACDD-1.3"
note = (
    f"{stamp}: repackaged for NODD from ERDDAP {DATASET_ID} monthly files; "
    f"rechunked to {tuple(CHUNKS[d] for d in ('time','mean_pressure','latitude','longitude'))} "
    f"(time, mean_pressure, latitude, longitude)."
)
ds.attrs["history"] = note + "\n" + ds.attrs.get("history", "")

# Repair any mojibake so h5netcdf can write the attributes (global + per-variable)
ds.attrs = {k: clean_utf8(v) for k, v in ds.attrs.items()}
for var in ds.variables:
    ds[var].attrs = {k: clean_utf8(v) for k, v in ds[var].attrs.items()}

# Sanity-check the key attributes
for k in ("title", "institution", "source", "references", "license", "Conventions"):
    print(f"{k}: {ds.attrs.get(k, '<missing>')}")

## Step 5 — Rechunk and set the on-disk encoding

Dask chunks (for streaming the write) and the netCDF `chunksizes` encoding are both set to
`CHUNKS`. The time chunk is capped at the block length so the last (short) block stays valid.

**CF:** a `_FillValue` is set on the data variable and explicitly suppressed on the coordinate
variables (CF discourages `_FillValue` on coordinates, but xarray adds it to float coords by
default).

**virtualizarr / icechunk (downstream):** this file layout is built to virtualize cleanly into
an Icechunk store later —

- every file uses the **same** chunk grid `(100, 1, 180, 180)`, so along `time` each file is a
  single time-chunk and the files concatenate onto one regular Zarr chunk grid (the final
  block's 70-step chunk is the allowed smaller last chunk);
- `zlib` (deflate) + `shuffle` are both representable as Zarr codecs, so the virtual references
  keep working with compression on;
- `dtype` and `_FillValue` are pinned identically across all files so the virtual dataset has
  one consistent fill value / dtype.

In [ ]:
# On-disk (physical) chunk sizes, in the variable's dimension order. These define the file's
# chunk grid for virtualizarr/icechunk downstream. The time chunk is capped at the block
# length so the final (short) block stays valid.
var_dims = ds[DATA_VAR].dims
chunksizes = tuple(min(CHUNKS[d], ds.sizes[d]) for d in var_dims)

# Dask chunks control how the data is streamed during the write. We deliberately keep them
# LARGER than the on-disk chunks — a full lat/lon plane per (time-block, pressure) — so each
# dask task reads a contiguous slab from the source files rather than many strided sub-tiles.
# This is memory-bounded (one plane-stack at a time, ~0.4 GB) and avoids the I/O thrash that
# a (time=100, mean_pressure=1, latitude=180, longitude=180) dask grid would cause on the
# unchunked source files. The dask chunks are exact multiples of `chunksizes`, so each on-disk
# chunk is written exactly once (no partial-chunk read-modify-write).
tchunk = min(CHUNKS["time"], ds.sizes["time"])
dask_chunks = {
    "time": tchunk,
    "mean_pressure": 1,
    "latitude": ds.sizes["latitude"],
    "longitude": ds.sizes["longitude"],
}
ds = ds.chunk(dask_chunks)

encoding = {
    DATA_VAR: {
        "chunksizes": chunksizes,
        "dtype": "float32",
        "_FillValue": np.float32(np.nan),   # CF: flag land/missing; keep identical across all files
        # Lossless per-chunk compression. shuffle + deflate level 4 is a good balance
        # for this ocean float32 data (lots of land/fill compresses well). Both filters
        # are representable as Zarr codecs, so they're fine for virtualizarr/icechunk.
        # Drop to complevel=1 if write time becomes the bottleneck during the full run.
        "zlib": True,
        "complevel": 4,
        "shuffle": True,
    },
    # Preserve the original time encoding
    "time": {"units": "seconds since 1970-01-01T00:00:00Z", "dtype": "float64", "_FillValue": None},
}
# CF: coordinate variables should not carry a _FillValue. xarray adds one to float
# coordinates by default, so suppress it explicitly on each coordinate.
for coord in ("mean_pressure", "latitude", "longitude"):
    encoding[coord] = {"_FillValue": None}

print("variable dims :", var_dims)
print("on-disk chunks:", chunksizes)
print("dask chunks   :", {d: dask_chunks[d] for d in var_dims})

## Step 6 — Write the output netCDF locally

In [ ]:
out_path = os.path.join(OUTPUT_DIR, block["filename"])
print("Writing", out_path, "...")
ds.to_netcdf(out_path, engine="h5netcdf", encoding=encoding, mode="w")
print(f"Done: {os.path.getsize(out_path) / 1e9:.2f} GB")

In [ ]:
# Verify the written file: chunking, compression, and CF attributes
check = xr.open_dataset(out_path, engine="h5netcdf")
enc = check[DATA_VAR].encoding
print("dims           :", dict(check.sizes))
print("stored chunking:", enc.get("chunksizes"))
print("compression    :", {k: enc.get(k) for k in ("zlib", "complevel", "shuffle")})
print("_FillValue     :", enc.get("_FillValue"))
print("Conventions    :", check.attrs.get("Conventions"))
print(f"{DATA_VAR}.standard_name:", check[DATA_VAR].attrs.get("standard_name"))
print("mean_pressure  :", {k: check["mean_pressure"].attrs.get(k) for k in ("standard_name", "positive", "axis")})
check.close()

## Step 7 — Upload the single file to NODD

Uses `gcsfs` with the application-default credentials (same approach that already works for
the `index.html` upload).

In [ ]:
fs = gcsfs.GCSFileSystem(token=GCS_TOKEN)

dest = f"{NODD_DEST}/{block['filename']}"
print(f"Uploading {out_path}\n       -> {dest}")
fs.put(out_path, dest)
print("Uploaded.")

In [ ]:
# Verify it landed
info = fs.info(dest)
print(dest.replace("gs://", ""))
print(f"  size: {info['size'] / 1e9:.2f} GB")

## Step 8 — (optional) Clean up the downloaded monthly files

The monthly source files are only needed to build the output. Remove them once the upload is
verified to free scratch space.

In [ ]:
# for f in local_files:
#     os.remove(f)
# print("Removed", len(local_files), "monthly source files")